In [2]:
import pandas as pd
import numpy as np

In [3]:
movies = pd.read_csv('tmdb_5000_movies.csv')
credits = pd.read_csv('tmdb_5000_credits.csv')

In [7]:
movies = movies.merge(credits, on = 'title')

In [11]:
movies = movies[['movie_id','title','overview','genres','keywords','cast','crew']]

In [14]:
movies.dropna(inplace=True)

In [16]:
movies.duplicated().sum()

np.int64(0)

In [19]:
import ast

In [20]:
def convert(obj):
    L=[]
    for i in ast.literal_eval(obj):
        L.append(i['name'])
    return L

In [21]:
movies['genres']=movies['genres'].apply(convert)

In [24]:
movies['keywords']= movies['keywords'].apply(convert)

In [29]:
def convert3(obj):
    L=[]
    counter = 0
    for i in ast.literal_eval(obj):
        if counter!= 3:
            L.append(i['name'])
            counter += 1
        else:
            break
    return L

In [30]:
movies['cast']= movies['cast'].apply(convert3)

In [34]:
def fetch_director(obj):
    L=[]
    for i in ast.literal_eval(obj):
        if i['job']=='Director':
            L.append(i['name'])
            break
    return L

In [35]:
movies['crew']= movies['crew'].apply(fetch_director)

In [37]:
movies['overview'] = movies['overview'].apply(lambda x: x.split())

In [39]:
movies['genres'] = movies['genres'].apply(lambda x:[i.replace(" ","") for i in x])

In [40]:
movies['keywords'] = movies['keywords'].apply(lambda x:[i.replace(" ","") for i in x])
movies['cast']= movies['cast'].apply(lambda x:[i.replace(" ","") for i in x])
movies['crew'] = movies['crew'].apply(lambda x:[i.replace(" ","") for i in x])


In [42]:
movies['tags'] = movies['overview'] + movies ['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

In [44]:
new_df = movies [['movie_id','title', 'tags']]

In [46]:
new_df['tags'] = new_df['tags'].apply(lambda x:" ".join(x))

C:\Users\ishan\AppData\Local\Temp\ipykernel_25952\3089450492.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(lambda x:" ".join(x))


In [49]:
new_df['tags']= new_df['tags'].apply(lambda x:x.lower())

C:\Users\ishan\AppData\Local\Temp\ipykernel_25952\899821970.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags']= new_df['tags'].apply(lambda x:x.lower())


In [50]:
new_df

,movie_id,title,tags
0,19995,Avatar,"in the 22nd century, a paraplegic marine is di..."
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believed to be dead, ha..."
2,206647,Spectre,a cryptic message from bond’s past sends him o...
3,49026,The Dark Knight Rises,following the death of district attorney harve...
4,49529,John Carter,"john carter is a war-weary, former military ca..."
...,...,...,...
4804,9367,El Mariachi,el mariachi just wants to play his guitar and ...
4805,72766,Newlyweds,a newlywed couple's honeymoon is upended by th...
4806,231617,"Signed, Sealed, Delivered","""signed, sealed, delivered"" introduces a dedic..."
4807,126186,Shanghai Calling,when ambitious new york attorney sam is sent t...


In [51]:
import nltk

In [52]:
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

In [53]:
def stem(text):
    y=[]
    for i in text.split():
        y.append(ps.stem(i))
    return " ".join(y)

In [54]:
new_df['tags'] = new_df['tags'].apply(stem)

C:\Users\ishan\AppData\Local\Temp\ipykernel_25952\3213734980.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(stem)


In [56]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=5000, stop_words='english')

In [57]:
vectors = cv.fit_transform(new_df['tags']).toarray()

In [60]:
from sklearn.metrics.pairwise import cosine_similarity

In [61]:
similarity = cosine_similarity(vectors)

In [63]:
 def recommend(movie):
     movie_index = new_df[new_df['title']== movie].index[0]
     distance = similarity[movie_index]
     movies_list = sorted(list(enumerate(distance)),reverse = True, key = lambda x:x[1])[1:6]
     for i in movies_list:
         print(new_df.iloc[i[0]].title)

In [64]:
recommend('Avatar')

Aliens vs Predator: Requiem
Aliens
Falcon Rising
Independence Day
Titan A.E.


In [68]:
recommend('Tangled')

Out of Inferno
The Princess and the Frog
Home on the Range
Animals United
Toy Story 3


In [69]:
recommend('Harry Potter and the Half-Blood Prince')

Harry Potter and the Order of the Phoenix
Harry Potter and the Goblet of Fire
Harry Potter and the Chamber of Secrets
Harry Potter and the Philosopher's Stone
Harry Potter and the Prisoner of Azkaban
